In [2]:
# embed a word into vector

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
# The first time you run this, it downloads the model (~80 MB) and the tokenizer from HuggingFace. 
# The tokenizer turns text into something the model can read. 
# After that, both load from a local cache.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [5]:
v1.shape

(384,)

In [6]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [7]:
v1.dot(dv) # vector multiplication

np.float32(0.32332397)

In [8]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [9]:
v2.dot(dv)

np.float32(0.019730438)

The all-MiniLM-L6-v2 model outputs normalized vectors - vectors with unit length. When both vectors are normalized, the dot product equals cosine similarity. That's why the model documentation says it "uses cosine similarity."

Cosine similarity measures the angle between two vectors, ignoring their length:

1.0 = same direction (similar)
0.0 = perpendicular (unrelated)
-1.0 = opposite direction (opposite meaning)
Formally, if theta is the angle between two vectors, cosine similarity is cos(theta):

cos(0) = 1 - vectors point in the same direction
cos(90) = 0 - vectors are perpendicular
cos(180) = -1 - vectors point in opposite directions
Because our vectors are normalized, the dot product gives us cosine similarity directly. This is why we can use v1.dot(dv) to compare texts.

<h2> Embedding Dataset </h2>

In [10]:
from ingest import load_faq_data

documents = load_faq_data()

In [11]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [7]:
texts[0]

"Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."

In [12]:
from tqdm.auto import tqdm

In [13]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [14]:
import numpy as np
X = np.array(vectors)

In [11]:
X.shape

(1350, 384)

<h2> Vector Search </h2>

In [3]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [12]:
scores = X.dot(v_query)
scores

array([ 0.48740575,  0.20991933,  0.762941  , ..., -0.08637968,
        0.03759793, -0.03037044], shape=(1350,), dtype=float32)

This is matrix-vector multiplication. Each element i of scores is the cosine similarity between document i (row i of X) and v_query.

In [13]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [14]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [15]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [16]:
# np.argsort sorts from lowest to highest, so the last 5 are the top ones:
top5 = np.argsort(scores)[-5:]

In [17]:
# reverse them to get the highest first
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [20]:
# There's a shorter trick I usually reach for. 
# We negate the scores first, so the largest becomes the smallest. 
# Then argsort puts the best matches at the front.
top5 = np.argsort(-scores)[:5]

TypeError: bad operand type for unary -: 'list'

In [18]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294106
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions'

<h2>Vector Search with minsearch </h2>

In [15]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [22]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [23]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [24]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [25]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

<h2> RAG with Vector Search </h2>

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

Below still uses keyword search. Text search isn't bad here, so the answer may already look right.

In [4]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

Text search takes the query string directly, but vector search needs the query as a vector first. So we subclass RAGBase and override search to encode the query before searching.

In [5]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [16]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [17]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. You can also start learning and submitting homework while the form is open. If you want a certificate, make sure to submit your project before submissions close.'

What we have done above is exact nearest neighbor (NN) search. We score the query against every document and pick the top ones. It always finds the true top matches, but it pays for that by touching everything.

Approximate nearest neighbor (ANN) search takes a shortcut. Instead of comparing against everything, it first narrows down to a region of likely matches. Then it scores only within that region. It may miss the absolute best match, but the results are still good and it's much faster.

NN (exact):    compare query against ALL documents -> top 5
ANN (approx):  narrow down to a region -> compare within region -> top 5

<h2> Persistent search - sqlitesearch</h2>

It stores vectors in SQLite, a real on-disk database, and uses ANN strategies for retrieval. Because the data lives on disk, one process can write the vectors and another can read them back.

sqlitesearch supports three ANN modes:

lsh (default): up to 100K vectors, random hyperplane projections
ivf: 10K-500K vectors, K-means clustering
hnsw: 10K-1M+ vectors, proximity graph (highest recall)


In [18]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [19]:
vs_index.fit(vectors, documents)

The index is saved to faq_vectors2.db. Unlike minsearch, this file persists on disk. You can search immediately after indexing, or reopen the index later without re-indexing.

In [20]:
uery = "I just discovered the course. Can I still join it?"
# encode query for vector search
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [21]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'data-engineering-zoomcamp',
  'section': 'G

In [22]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [23]:
vs_index.close()

<h2> Reopening and reuse embedding and index </h2>

In [24]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

In [26]:
results

[{'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```',
  'doc_id': '5ca6890c1a'},
 {'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BOOTSTRAP_SERVERS_CO